In [13]:
import os

os.chdir("C:/Online Courses/NYP/GenAIandDeepL/genAI-deepL-project")
print(os.getcwd())


C:\Online Courses\NYP\GenAIandDeepL\genAI-deepL-project


In [14]:
import os
import pandas as pd

test_df = pd.read_csv("./data/test_df.csv")

# Normalize paths (important for Windows)
def extract_relative_path(path):
    path = path.replace("\\", "/")

    # Split by "Data/"
    if "Data/" in path:
        return path.split("Data/")[1]

    return os.path.basename(path)

# Build allowed relative path set
test_df["relative_image_path"] = test_df["image_path"].apply(
    extract_relative_path
)

# Sample 5 images per engagement_label (or all if <5 exist)
sampled_df = (
    test_df.groupby("engagement_label", group_keys=False)
    .apply(lambda x: x.sample(min(len(x), 5), random_state=42))
)

# Keep only 15 images total (5 per class if 3 classes exist)
allowed_paths = set(sampled_df["relative_image_path"])


C:\Users\sugar\AppData\Local\Temp\ipykernel_59172\1682992069.py:24: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), 5), random_state=42))


Filtering test df for small version

In [16]:
import pandas as pd

# Load full dataset (if not already loaded)
df = pd.read_csv("./data/test_df.csv")

# Normalize image_path the same way as before
def extract_relative_path(path):
    path = path.replace("\\", "/")

    if "Data/" in path:
        return path.split("Data/")[1]

    return os.path.basename(path)

# Create relative path column for matching
df["relative_image_path"] = df["image_path"].apply(extract_relative_path)

# Filter rows that are in allowed_paths
filtered_df = df[df["relative_image_path"].isin(allowed_paths)]

# Save to new CSV
filtered_df.to_csv("./data/test_df_small.csv", index=False)

print(len(filtered_df))


15


Filtering images for small version

In [15]:
import shutil

# Folder setup
source_root = "datasets/Data"
output_root = "datasets/test_subset_small"

os.makedirs(output_root, exist_ok=True)

# Walk through dataset folder
for root, _, files in os.walk(source_root):
    for file in files:

        # Construct relative path from Data folder
        full_path = os.path.join(root, file)

        rel_path = os.path.relpath(full_path, source_root)
        rel_path = rel_path.replace("\\", "/")

        if rel_path in allowed_paths:
            dest_path = os.path.join(output_root, rel_path)

            os.makedirs(os.path.dirname(dest_path), exist_ok=True)

            shutil.copy2(full_path, dest_path)
